# Week 2 Day 4

## Covered Today
1. How LLM Tool Calling Really Works (No Magic, Just Prompts)
2. Common Use Cases for LLM Tools and Agentic AI Workflows
3. Building an Airline AI Assistant with Tool Calling in OpenAI and Gradio
4. Handling Multiple Tool Calls with OpenAI and Gradio
5. Building Tool Calling with SQLite Database Integration

In [1]:
# we start by setting up the entire boilerplate for AI LLM interacion


# required imports
import os
from dotenv import load_dotenv
import requests
from openai import OpenAI
from IPython.display import Markdown, display, update_display
import gradio as gr 


# load API Keys
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')


# check if all loaded keys exist and are as per the format
if openai_api_key:
    if openai_api_key.startswith("sk-"):
        print(f"OpenAI      : OK           (begins {openai_api_key[:7]}...)")
    else:
        print("OpenAI      : WRONG FORMAT (should start with 'sk-')")
else:
    print("OpenAI      : MISSING")

if anthropic_api_key:
    if anthropic_api_key.startswith("sk-ant-"):
        print(f"Anthropic   : OK           (begins {anthropic_api_key[:10]}...)")
    else:
        print("Anthropic   : WRONG FORMAT (should start with 'sk-ant-')")
else:
    print("Anthropic   : MISSING")

if google_api_key:
    if google_api_key.startswith("AQ.Ab"):
        print(f"Google      : OK           (begins {google_api_key[:5]}...)")
    else:
        print("Google      : WRONG FORMAT (should start with 'AQ.Ab' or 'AIza')")
else:
    print("Google      : MISSING")

if openrouter_api_key:
    if openrouter_api_key.startswith("sk-or-"):
        print(f"OpenRouter  : OK           (begins {openrouter_api_key[:8]}...)")
    else:
        print("OpenRouter  : WRONG FORMAT (should start with 'sk-or-')")
else:
    print("OpenRouter  : MISSING")


# create clients for each provider
openai_client = OpenAI()
google_url = 'https://generativelanguage.googleapis.com/v1beta/openai/'
anthropic_url = 'https://api.anthropic.com/v1/'
openrouter_url = 'https://openrouter.ai/api/v1'
ollama_url = 'http://127.0.0.1:11434/v1'


google_client = OpenAI(base_url=google_url, api_key=google_api_key)
anthropic_client = OpenAI(base_url=anthropic_url, api_key=anthropic_api_key)
openrouter_client = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
ollama_client = OpenAI(base_url=ollama_url, api_key='Ollama')

OpenAI      : OK           (begins sk-proj...)
Anthropic   : OK           (begins sk-ant-api...)
Google      : OK           (begins AQ.Ab...)
OpenRouter  : OK           (begins sk-or-v1...)


In [2]:
# Now we move ahead with setting up our airline assistant
system_message = "You are a helpful assistant for an airline called flighty. Give short, courteous answers, no more than 1 sentence. Always be accurate. If you are not aware of any information, politely inform the user so."

In [3]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]

    response = openai_client.chat.completions.create(model='gpt-5-mini', messages=messages) #type: ignore
    
    return response.choices[0].message.content


In [4]:
gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [5]:
ticket_prices = {"london": "$1999", "madrid": "$999", "chennai": "$199", "mumbai": "$299", "toronto": "$1499"}

In [7]:
def get_ticket_price(destination_city):
    print(f"Tool called for city: {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of the ticket to {destination_city} is {price}."

In [13]:
get_ticket_price("london")
# so far, we have just created a dict with cities and their prices and then we have created a function to get the price from the list. Now, in order to pass on this information to the LLM, we use a fixed json format, where we describe this function

Tool called for city: london


'The price of the ticket to london is $1999.'

In [ ]:
# Now, we can not provide the above function to the LLM directly, we need to pass on the information of the function in json format. For this, we save the json in a variable. 

# Let's start by declaring the variable and giving it an empty dict (format used for json)
price_function = {}

# next, we add the name of the function, which will be acting as a tool.
price_function = {
    "name": "get_ticket_price"
}

In [15]:
# Once the name is passed, we move on to pass on the description of the function, here, we have to be completely unambigous, since, this description will actually tell the LLM, what this function does. In our case, this function simply gives the price of the return ticket to a destiation city. So, we add this information in the function.
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city"
}

In [21]:
# Now, a function also contains parameters and to make this function run, we would need arguments to replace the parameters while running these functions, so for that we would need to also explain these parameters to the function. Now, parameter information in this json is being passed on as a dict, which is also called Object in json language. So, we simply add, parameters as a key and in this open a new dict and pass on the type as object. This also tells the LLM that you should also, return the information in the form of the object. As a reply to this tool call, the LLM would return {"destination_city": "london"}

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city",
    "parameters": {
        "type": "object"
    }
}

In [22]:
# Now to explain each parameter to be passed on in this function, we add a new key, properties, which can be used to cover all the parameters of this function. Now, once we add the properties key, we add a new dict to it, which will contain the information of each paramter.

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city",
    "parameters": {
        "type": "object",
        "properties": {
            
        }
    }
}

In [26]:
# now we pass on each paramter one by one, describe its type and give its description.

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The destination city, or the city that the customer wants to travel to."
            }
        }
    }
}

# under properties, the way we have added destination_city, we can add more parameters if the function has more parameters. for example, if the function also takes in num_pax(number of passengers, then we can pass on this also as below)
"""
"parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The destination city, or the city that the customer wants to travel to."
            },
            "num_pax": {
                "type": "integer",
                "description": "Total count of passengers travelling to this destination city."
            }
        }
    }
"""
# since, our function only wants one parameter, we keep it as is

'\n"parameters": {\n        "type": "object",\n        "properties": {\n            "destination_city": {\n                "type": "string",\n                "description": "The destination city, or the city that the customer wants to travel to."\n            },\n            "num_pax": {\n                "type": "integer",\n                "description": "Total count of passengers travelling to this destination city."\n            }\n        }\n    }\n'

In [27]:
# once all the parameters are defined, we inform the LLM if there is any paramters which is required in order to run the function. in our case, it is destination_city. we add a new "required" key iun the parameters and pass on a list of required parameters.

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The destination city, or the city that the customer wants to travel to."
            }
        },
        "required": ["destination_city"]
    }
}

In [28]:
# the LLM is as we know a token predicter, hence, it can also pass in information back that is not needed. for example, if the user tells it is for 2 passengers, but we are not passing this information to the function. If that additional information comes across, we we will get an error while running the function. so, we add another key in parameters by the name of additionalProperties (camel case) and mark its value as False. There are two more values that can be passed, but we will stick to false for now. 

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The destination city, or the city that the customer wants to travel to."
            }
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

# this marks the completion of this json